# Exercise 2 — Fine-tune for Binary Text Classification
Các bước thực hiện: cài thư viện, tải IMDb, khởi tạo DistilBERT, tokenize, cấu hình Trainer, fine-tune và đánh giá. Thực hiện EDA trong `01_eda_pca.ipynb` trước khi huấn luyện.

## 1. Install the necessary libraries

In [ ]:
%pip install -q "transformers>=4.45,<5" "datasets>=3,<5" "evaluate>=0.4" "accelerate>=1,<2" scikit-learn torch

## 2. Load IMDb binary text classification dataset

In [ ]:
import json
from pathlib import Path
import sys
project_root = Path.cwd()
if not (project_root / "configs").exists(): project_root = project_root.parent
sys.path.insert(0,str(project_root))
from src.data import prepare_splits
cfg = json.loads((project_root / "configs/default.json").read_text(encoding="utf-8"))
train, val, test = prepare_splits(cfg)
print("Train",len(train),"Validation",len(val),"Test",len(test))
print("Labels",set(train["label"]))

## 3. Load pretrained model and tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained(cfg["model"])
model = AutoModelForSequenceClassification.from_pretrained(cfg["model"],num_labels=2)

## 4. Preprocess the dataset

In [ ]:
from transformers import DataCollatorWithPadding
def tokenize(batch):
    return tokenizer(batch["text"],truncation=True,max_length=cfg["max_length"])
train_tok = train.map(tokenize,batched=True)
val_tok = val.map(tokenize,batched=True)
test_tok = test.map(tokenize,batched=True)
collator = DataCollatorWithPadding(tokenizer)

## 5. Define training arguments

In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(output_dir=str(project_root / "outputs/notebook_run"), num_train_epochs=cfg["epochs"], per_device_train_batch_size=cfg["batch_size"], per_device_eval_batch_size=cfg["batch_size"], learning_rate=cfg["learning_rate"], weight_decay=cfg["weight_decay"], eval_strategy="epoch",save_strategy="epoch",load_best_model_at_end=True,metric_for_best_model="eval_loss",greater_is_better=False,report_to="none")

## 6. Create a Trainer object and fine-tune the model
**Cảnh báo:** Cell kế tiếp tốn thời gian và sẽ cập nhật trọng số.

In [ ]:
from transformers import Trainer
from src.train import metrics
trainer = Trainer(model=model,args=training_args,train_dataset=train_tok,eval_dataset=val_tok,processing_class=tokenizer,data_collator=collator,compute_metrics=metrics)
train_result = trainer.train()
print("Best checkpoint:",trainer.state.best_model_checkpoint)
print("Best val loss:",trainer.state.best_metric)

## 7. Evaluate the fine-tuned model
Chỉ đánh giá test sau khi checkpoint đã được chọn bằng validation.

In [ ]:
result = trainer.predict(test_tok)
print(result.metrics)